In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys
import os
import pickle
import logging
import warnings
import re
from typing import Dict, List
from pathlib import Path
import json 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import matplotlib.gridspec as gridspec
import geopandas as gpd

import torch
import massbalancemachine as mbm

sys.path.append(os.path.join(os.getcwd(), '../../'))
from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.config_models import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.device_count() == 0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"FD_MODEL"
CACHE_DIR = BASE_DIR / "datasets/"

# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)

# make sure BASE_DIR exists
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Base directory for data: {BASE_DIR}")
print(f"Cache directory for models: {CACHE_DIR}")

BARPLOT_ALPHA = 0.7

In [ ]:
def to_roman(n):
    vals = [(1000, 'm'), (900, 'cm'), (500, 'd'), (400, 'cd'), (100, 'c'),
            (90, 'xc'), (50, 'l'), (40, 'xl'), (10, 'x'), (9, 'ix'), (5, 'v'),
            (4, 'iv'), (1, 'i')]
    result = ''
    for v, s in vals:
        while n >= v:
            result += s
            n -= v
    return result

## Datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (SJM+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
print(num_measurements)

SOURCE_REGIONS = ['CH', 'NOR', 'ISL', "FR"]
TARGET_REGIONS = ['IT_AT', 'SJM', 'CENTRALASIA', 'ALA']

run_flag_by_code = {
    'ALA': False,
    'CENTRALASIA': False,
    'CH': False,
    'ISL': False,
    'NOR': False,
    'SJM': False,
    'FR': False,
    'IT_AT': False
}
# Compute monthly data once per code
monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=VOIS_CLIMATE,
    vois_topographical=STATIC_COLS,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

### Subregions:

In [ ]:
region_colors = [
    "red", "blue", "green", "purple", "orange", "darkred", "cadetblue",
    "darkgreen", "darkpurple", "pink", "gray", "black"
]

In [ ]:
# Central Asia
dfs, glacier_dict_ca = add_o2region_to_dfs(dfs, '13', cfg)

# Map O2Region into monthly_cache for CENTRALASIA
monthly_cache['CENTRALASIA']['data_monthly']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))
monthly_cache['CENTRALASIA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly_aug']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))

# Central Asia subregions (O2Region already mapped onto dfs['13'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'CENTRALASIA',
    subregions={
        'CA_12': {
            'o2region_col': 'O2Region',
            'values': ['1', '2']
        },
        'CA_3': {
            'o2region_col': 'O2Region',
            'values': ['3']
        },
        'CA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
    },
)

o2_color_map = {
    r: region_colors[i % len(region_colors)]
    for i, r in enumerate(
        sorted(v['O2Region'] for v in glacier_dict_ca.values()
               if v['O2Region'] is not None))
}
color_map = {
    g: o2_color_map[info['O2Region']]
    for g, info in glacier_dict_ca.items() if info['O2Region'] is not None
}
m = plot_stakes_folium(dfs['13'], color_map=color_map, tooltip_col='O2Region')
m

In [ ]:
# Alaska
dfs, glacier_dict_alaska = add_o2region_to_dfs(dfs,
                                               '01',
                                               cfg,
                                               deduplicate_glaciers=True)

# Map O2Region into monthly_cache for ALA
monthly_cache['ALA']['data_monthly']['O2Region'] = (
    monthly_cache['ALA']['data_monthly']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))
monthly_cache['ALA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['ALA']['data_monthly_aug']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))

# Alaska subregions (O2Region already mapped onto dfs['01'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'ALA',
    subregions={
        'ALA_2': {
            'o2region_col': 'O2Region',
            'values': ['2']
        },
        'ALA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
        'ALA_6': {
            'o2region_col': 'O2Region',
            'values': ['6']
        },
    },
)

o2_color_map = {
    r: region_colors[i % len(region_colors)]
    for i, r in enumerate(
        sorted(v['O2Region'] for v in glacier_dict_alaska.values()
               if v['O2Region'] is not None))
}
color_map = {
    g: o2_color_map[info['O2Region']]
    for g, info in glacier_dict_alaska.items() if info['O2Region'] is not None
}

m = plot_stakes_folium(dfs['01'], color_map=color_map, tooltip_col='O2Region')
m

In [ ]:
# IT/AT split
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'IT_AT',
    subregions={
        'IT': IT_GLACIERS,
        'AT': AT_GLACIERS
    },
    drop_parent=True,
)

# With subregions
TARGET_REGIONS_SUB = [
    'IT', 'AT', 'SJM', 'CA_3', 'CA_4', 'ALA_2', 'ALA_4', 'ALA_6'
    #'CENTRALASIA', 'ALA', #'CA_12',
]

XREG_PAIRS = [(src,
               [r for r in SOURCE_REGIONS + TARGET_REGIONS_SUB if r != src])
              for src in SOURCE_REGIONS]

# Assemble train/test pairs from cache — no ERA5 reprocessing
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

months_head_pad = res_xreg_by_source["CH"]['months_head_pad']
months_tail_pad = res_xreg_by_source["CH"]['months_tail_pad']
print(f"months_head_pad: {months_head_pad}")
print(f"months_tail_pad: {months_tail_pad}")

## Model assets:

#### Pooled set: 
Pool source regions together

In [ ]:
# Build glacier→region mapping from all source regions (no ranking needed)
df_all_glaciers = pd.concat(
    [
        res_xreg_by_source[region]["df_train"][[
            "GLACIER"
        ]].drop_duplicates().assign(region=region) for region in SOURCE_REGIONS
    ],
    ignore_index=True,
).rename(columns={"GLACIER": "glacier"})

# Add measurement counts (needed by build_assets_from_glacier_list internally? No — only used by build_glacier_subsets)
# df_ranked just needs columns: "glacier", "region"  ← that's all build_assets uses
glaciers_all = df_all_glaciers["glacier"].tolist()
print(f"Total unique glaciers across all source regions: {len(glaciers_all)}")

assets_full = build_assets_from_glacier_list(
    glaciers=glaciers_all,
    df_ranked=df_all_glaciers,  # acts as the gl→region lookup
    res_xreg_by_source=res_xreg_by_source,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    cache_path=str(CACHE_DIR / "assets_foundation_100pct.joblib"),
    force_recompute=False,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    src_regions=SOURCE_REGIONS,
)

# Failsafe: check that pooled training data only contains source region codes
actual_codes = set(df_all_glaciers.region.unique())
expected_codes = set(SOURCE_REGIONS)
unexpected = actual_codes - expected_codes
if unexpected:
    raise ValueError(
        f"Pooled training set contains unexpected SOURCE_CODEs: {unexpected}")
else:
    print(f"✓ Pooled set SOURCE_CODEs OK: {actual_codes}")

print(f"Pooled model assets: {len(assets_full['ds_train'])} sequences")

#### Test (hold-out) regions:

In [ ]:
# ── Always compute scalers/blurs — needed for any recomputation ───────────────
res_source_only = {
    region: {
        "df_train": res_xreg_by_source[region]["df_train"],
    }
    for region in SOURCE_REGIONS
}

scaler_m, scaler_s, scaler_joint = build_global_scalers_multi_source_simple(
    res_source_only,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
)
blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
    res_source_only,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)
print(
    f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
)

In [ ]:
RECOMPUTE_SPLITS = False

FT_FRAC = 0.10
save_path = BASE_DIR / "glacier_splits_FD.json"

#gl_level_split = ['IT', 'AT']
gl_level_split = []
id_level_split = ['IT', 'AT', 'SJM', 'CA_3', 'CA_4', 'ALA_2', 'ALA_4', 'ALA_6']

# ── Load existing glacier splits from disk ────────────────────────────────────
splits = {}
if not RECOMPUTE_SPLITS and save_path.exists():
    with open(save_path, "r") as f:
        splits = json.load(f)
    print(f"Loaded splits from: {save_path}")

# ── Only compute glacier splits for regions that need them ────────────────────
regions_needing_gl_split = [r for r in gl_level_split]
regions_to_compute = (regions_needing_gl_split if RECOMPUTE_SPLITS else
                      [r for r in regions_needing_gl_split if r not in splits])

for region in regions_to_compute:
    print(f"\n{'='*50}\nSplitting region: {region}")
    split = split_pool_holdout_sinkhorn(
        df_region=monthly_cache[region]["data_monthly"],
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        blur_joint=blur_joint,
        pool_frac=0.2,
        n_restarts=50,
        seed=cfg.seed,
    )
    splits[region] = split
    print(f"  Pool glaciers    : {split['pool_glaciers']}")
    print(f"  Holdout glaciers : {split['holdout_glaciers']}")
    print(f"  Pool fraction    : {split['actual_pool_frac']:.1%}")
    print(f"  Sk(pool↔holdout) : {split['sinkhorn_pool_vs_holdout']:.4f}")

if regions_to_compute:
    with open(save_path, "w") as f:
        json.dump(splits, f, indent=2)
    print(f"\nSaved splits for: {regions_to_compute}")

In [ ]:
FORCE_RECOMPUTE = False
FORCE_RECOMPUTE_REGIONS = []

N_REPEATS = 5

# ── Build FT assets for all target subregions, repeated N_REPEATS times ──────
ft_assets_glacier = {}
ft_assets_id = {region: [] for region in TARGET_REGIONS_SUB}

for region in TARGET_REGIONS_SUB:
    print(f"\n{'='*50}\nBuilding FT assets for: {region}")

    df_monthly = monthly_cache[region]["data_monthly"]
    df_monthly_aug = monthly_cache[region]["data_monthly_aug"]
    head_pad = monthly_cache[region]["months_head_pad"]
    tail_pad = monthly_cache[region]["months_tail_pad"]
    force = FORCE_RECOMPUTE or (region in FORCE_RECOMPUTE_REGIONS)

    # 1. Glacier-level split (unchanged — deterministic by glacier, no repeats needed)
    if region in gl_level_split:
        if region in gl_level_split:
            pool_glaciers = splits[region]["pool_glaciers"]
            holdout_glaciers = splits[region]["holdout_glaciers"]
            exp_key = f"FD_to_{region}"

            ds_ft_gl = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key}_FT",
                df_loss=df_monthly[df_monthly["GLACIER"].isin(pool_glaciers)],
                df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                    pool_glaciers)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="ft",
                show_progress=False,
            )
            ds_test_gl = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key}_TEST",
                df_loss=df_monthly[df_monthly["GLACIER"].isin(
                    holdout_glaciers)],
                df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                    holdout_glaciers)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="test",
                show_progress=False,
            )
            ft_train_idx_gl, ft_val_idx_gl = mbm.data_processing.MBSequenceDataset.split_indices(
                len(ds_ft_gl), val_ratio=0.2, seed=cfg.seed)
            ft_assets_glacier[region] = {
                "ds_ft": ds_ft_gl,
                "ds_test": ds_test_gl,
                "ft_train_idx": ft_train_idx_gl,
                "ft_val_idx": ft_val_idx_gl,
            }
            print(
                f"  [glacier-split] ds_ft  : {len(ds_ft_gl)} seqs | ds_test: {len(ds_test_gl)} seqs"
            )

    # 2. ID-level split, repeated N_REPEATS times with different seeds
    if region in id_level_split:
        winter_ids_full = df_monthly[df_monthly["PERIOD"] ==
                                     "winter"]["ID"].unique().tolist()
        annual_ids_full = df_monthly[df_monthly["PERIOD"] ==
                                     "annual"]["ID"].unique().tolist()

        for rep in range(N_REPEATS):
            winter_ids = list(winter_ids_full)
            annual_ids = list(annual_ids_full)

            rep_seed = cfg.seed + rep
            rng = np.random.default_rng(rep_seed)
            rng.shuffle(winter_ids)
            rng.shuffle(annual_ids)

            n_ft_w = max(1, int(len(winter_ids) *
                                FT_FRAC)) if winter_ids else 0
            n_ft_a = max(1, int(len(annual_ids) * FT_FRAC))
            ft_ids = winter_ids[:n_ft_w] + annual_ids[:n_ft_a]
            test_ids = winter_ids[n_ft_w:] + annual_ids[n_ft_a:]

            # keep track of ft_ids per rep for the cross-rep check below
            if rep == 0:
                _ft_id_sets_for_check = []
            _ft_id_sets_for_check.append(set(ft_ids))

            exp_key_id = f"FD_to_{region}_IDsplit_rep{rep}"
            ds_ft_id = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key_id}_FT",
                df_loss=df_monthly[df_monthly["ID"].isin(ft_ids)],
                df_full=df_monthly_aug[df_monthly_aug["ID"].isin(ft_ids)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="ft",
                show_progress=False,
            )
            ds_test_id = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=f"{exp_key_id}_TEST",
                df_loss=df_monthly[df_monthly["ID"].isin(test_ids)],
                df_full=df_monthly_aug[df_monthly_aug["ID"].isin(test_ids)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force,
                kind="test",
                show_progress=False,
            )
            ft_train_idx_id, ft_val_idx_id = mbm.data_processing.MBSequenceDataset.split_indices(
                len(ds_ft_id), val_ratio=0.2, seed=rep_seed)

            ft_assets_id[region].append({
                "rep": rep,
                "seed": rep_seed,
                "ds_ft": ds_ft_id,
                "ds_test": ds_test_id,
                "ft_train_idx": ft_train_idx_id,
                "ft_val_idx": ft_val_idx_id,
            })
            print(f"  [ID-split rep {rep}] ds_ft  : {len(ds_ft_id)} seqs | "
                  f"ds_test: {len(ds_test_id)} seqs")

            # ── sanity check after all reps for this region are built ────────────
        if region in id_level_split:
            n_reps = len(_ft_id_sets_for_check)
            identical_pairs = [
                (i, j) for i in range(n_reps) for j in range(i + 1, n_reps)
                if _ft_id_sets_for_check[i] == _ft_id_sets_for_check[j]
            ]
            if identical_pairs:
                print(
                    f"  WARNING [{region}]: identical FT id-sets for repeats {identical_pairs}"
                )
            else:
                overlaps = [
                    len(_ft_id_sets_for_check[i] & _ft_id_sets_for_check[j]) /
                    len(_ft_id_sets_for_check[i]) for i in range(n_reps)
                    for j in range(i + 1, n_reps)
                ]
                print(
                    f"  [check] {region}: {n_reps} repeats distinct (mean overlap {np.mean(overlaps):.1%})"
                )

# convenience dict: glacier-split where available, ID-split (list of reps) for the rest
ft_assets = {
    **ft_assets_glacier,
    **{
        r: ft_assets_id[r]
        for r in ft_assets_id if r not in ft_assets_glacier
    },
}

## Transformer:

### Zero shot:

In [ ]:
TRAIN_FLAG = False
models_dir = BASE_DIR / "models" / "foundation"
os.makedirs(models_dir, exist_ok=True)

model_pooled, model_path, info = train_or_load_one_source_model(
    cfg=cfg,
    key="foundation_100pct",
    lstm_assets=assets_full,
    best_params=PARAMS_TRANSFORMER,
    device=device,
    models_dir=models_dir,
    prefix="transformer_foundation",
    train_flag=TRAIN_FLAG,
    force_retrain=False,
    epochs=N_EPOCHS,
    date=MODEL_DATE,
    model_type="transformer",
    verbose=True,
)

print(f"Pooled model saved to: {model_path}")

# Build scaler once outside the loop
ds_pooled_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
    assets_full["ds_train"])
ds_pooled_scaler.make_loaders(
    train_idx=assets_full["train_idx"],
    val_idx=assets_full["val_idx"],
    fit_and_transform=True,
    seed=cfg.seed,
    verbose=False,
)

print(f"\n{'='*60}")
print("Zero-shot evaluation of pooled model on target regions")
print(f"{'='*60}")

# ── glacier-split: single dict per region, unchanged ──────────────────────────
print(f"\n--- glacier-split ---")
for region in gl_level_split:
    ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ft_assets_glacier[region]["ds_test"])
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_test_copy, ds_pooled_scaler, batch_size=128, seed=cfg.seed)
    metrics, _ = model_pooled.evaluate_with_preds(device, test_dl,
                                                  ds_test_copy)
    print(f"  {region:15s}  RMSE_a={metrics['RMSE_annual']:.3f}  "
          f"RMSE_w={metrics['RMSE_winter']:.3f}  "
          f"R2_a={metrics['R2_annual']:.3f}  "
          f"R2_w={metrics['R2_winter']:.3f}")

# ── ID-split: list of reps per region, aggregate across reps ─────────────────
print(f"\n--- ID-split (mean ± std over {N_REPEATS} repeats) ---")
zs_id_metrics = {
}  # store per-region rep metrics for later reuse (e.g. plotting)

for region in id_level_split:
    rep_metrics = []
    for rep_data in ft_assets_id[region]:
        ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            rep_data["ds_test"])
        test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
            ds_test_copy, ds_pooled_scaler, batch_size=128, seed=cfg.seed)
        metrics, _ = model_pooled.evaluate_with_preds(device, test_dl,
                                                      ds_test_copy)
        rep_metrics.append(metrics)

    zs_id_metrics[region] = rep_metrics

    rmse_a = [m["RMSE_annual"] for m in rep_metrics]
    rmse_w = [m["RMSE_winter"] for m in rep_metrics]
    r2_a = [m["R2_annual"] for m in rep_metrics]
    r2_w = [m["R2_winter"] for m in rep_metrics]

    print(f"  {region:15s}  "
          f"RMSE_a={np.mean(rmse_a):.3f}±{np.std(rmse_a):.3f}  "
          f"RMSE_w={np.mean(rmse_w):.3f}±{np.std(rmse_w):.3f}  "
          f"R2_a={np.mean(r2_a):.3f}±{np.std(r2_a):.3f}  "
          f"R2_w={np.mean(r2_w):.3f}±{np.std(r2_w):.3f}")

In [ ]:
XREG_MODELS_BASE = Path(cfg.dataPath) / path_cache / "CROSS_REGION/models"
TRAIN_FLAG = False

single_source_models = {}
single_source_assets = {}
single_source_scalers = {}

for src in SOURCE_REGIONS:
    ckpt = XREG_MODELS_BASE / src / f"transformer_xreg_{src}_{src}_{MODEL_DATE}.pt"
    os.makedirs(ckpt.parent, exist_ok=True)

    df_src = res_xreg_by_source[src]["df_train"][["GLACIER"]] \
               .drop_duplicates().assign(region=src) \
               .rename(columns={"GLACIER": "glacier"})
    assets_src = build_assets_from_glacier_list(
        glaciers=df_src["glacier"].tolist(),
        df_ranked=df_src,
        res_xreg_by_source=res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        cache_path=str(XREG_MODELS_BASE / src / f"assets_{src}_100pct.joblib"),
        force_recompute=False,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        src_regions=[src],
    )
    single_source_assets[src] = assets_src

    # build per-source scaler
    scaler_src = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        assets_src["ds_train"])
    scaler_src.make_loaders(
        train_idx=assets_src["train_idx"],
        val_idx=assets_src["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )
    single_source_scalers[src] = scaler_src

    m = mbm.models.Transformer_MB.build_model_from_params(cfg,
                                                          PARAMS_TRANSFORMER,
                                                          device,
                                                          verbose=False)
    if ckpt.exists():
        state = torch.load(ckpt, map_location=device)
        m.load_state_dict(state)
        print(f"Loaded {src} model from {ckpt}")
    else:
        print(f"No checkpoint found for {src}, training from scratch...")
        m, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key=src,
            lstm_assets=assets_src,
            best_params=PARAMS_TRANSFORMER,
            device=device,
            models_dir=XREG_MODELS_BASE / src,
            prefix="transformer_xreg",
            train_flag=TRAIN_FLAG,
            force_retrain=False,
            epochs=N_EPOCHS,
            date=MODEL_DATE,
            model_type="transformer",
            verbose=False,
        )
        print(f"Trained {src} model, saved to {ckpt}")

    single_source_models[src] = m

eval_configs = []
for region in TARGET_REGIONS_SUB:
    ds_test_full = build_or_load_lstm_dataset_only(
        cfg=cfg,
        key=f"ZEROSHOT_{region}_FULL",
        df_loss=monthly_cache[region]["data_monthly"],
        df_full=monthly_cache[region]["data_monthly_aug"],
        months_head_pad=monthly_cache[region]["months_head_pad"],
        months_tail_pad=monthly_cache[region]["months_tail_pad"],
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=str(CACHE_DIR),
        force_recompute=False,
        kind="test",
        show_progress=False,
    )
    eval_configs.append((region, {"ds_test": ds_test_full}))


def _eval_on(model, ds_test, scaler):
    ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_test)
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_copy, scaler, batch_size=128, seed=cfg.seed)
    return model.evaluate_with_preds(device, test_dl, ds_copy)


df_metrics_comparison = {}
for tgt_label, assets in eval_configs:
    if assets is None:
        print(f"No assets for {tgt_label}, skipping.")
        continue

    df_metrics_comparison[tgt_label] = {}
    m, _ = _eval_on(model_pooled, assets["ds_test"], ds_pooled_scaler)
    df_metrics_comparison[tgt_label]["pooled_zs"] = m

    for src, model in single_source_models.items():
        m, _ = _eval_on(model, assets["ds_test"], single_source_scalers[src])
        df_metrics_comparison[tgt_label][src] = m

# --- print summary ---
print(f"\n{'='*80}")
print(f"{'Model':<15} {'RMSE_a':>8} {'RMSE_w':>8} {'R2_a':>8} {'R2_w':>8}")
print(f"{'='*80}")
for tgt_label, results in df_metrics_comparison.items():
    print(f"\n--- {tgt_label} ---")
    for model_label, m in results.items():
        print(f"  {model_label:<15}  "
              f"RMSE_a={m['RMSE_annual']:.3f}  "
              f"RMSE_w={m['RMSE_winter']:.3f}  "
              f"R2_a={m['R2_annual']:.3f}  "
              f"R2_w={m['R2_winter']:.3f}")

#### Barplot:

##### Annual:

In [ ]:
# glacier counts per region, from the data overview table
REGION_GLACIER_COUNTS = {
    "ALA_2": 8,
    "ALA_4": 2,
    "ALA_6": 2,
    "ISL": 55,
    "SJM": 3,
    "NOR": 32,
    "CH": 30,
    "FR": 9,
    "IT": 14,
    "AT": 10,
    "CA_1": 3,
    "CA_2": 2,
    "CA_3": 8,
    "CA_4": 2,
}

In [ ]:
# --- bar plot ---
MODEL_ORDER = ["pooled_zs"] + SOURCE_REGIONS
LABELS = {src: f"{src} only" for src in SOURCE_REGIONS}
LABELS["pooled_zs"] = "Pooled"
METRICS = [
    ("RMSE_annual", "RMSE annual (m w.e.)", False),
]

MAX_COLS = 4

EXCLUDE_REGIONS = {'CENTRALASIA', 'ALA'}
tgt_labels = [
    t for t in df_metrics_comparison.keys() if t not in EXCLUDE_REGIONS
]
n_tgts = len(tgt_labels)
n_metrics = len(METRICS)
n_cols = min(MAX_COLS, n_tgts)
n_rows_per_metric = math.ceil(n_tgts / n_cols)
n_rows = n_metrics * n_rows_per_metric

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(3.5 * n_cols, 3.8 * n_rows),
    sharey=False,
)
axes = np.array(axes).reshape(n_rows, n_cols)

for metric_row, (metric_key, metric_label, is_r2) in enumerate(METRICS):
    for tgt_idx, tgt_label in enumerate(tgt_labels):
        sub_row = tgt_idx // n_cols
        col = tgt_idx % n_cols
        row = metric_row * n_rows_per_metric + sub_row
        ax = axes[row, col]

        # # subplot label
        # ax.text(
        #     0.03,
        #     0.97,
        #     f"({chr(ord('a') + tgt_idx)})",
        #     transform=ax.transAxes,
        #     fontsize=NATURE_SPECS["font_max_pt"],
        #     #fontweight="bold",
        #     va="top",
        #     ha="left",
        #     zorder=5)

        ax.text(0.03,
                0.97,
                f"({to_roman(tgt_idx + 1)})",
                transform=ax.transAxes,
                fontsize=NATURE_SPECS["font_max_pt"],
                va="top",
                ha="left",
                zorder=5)

        results = df_metrics_comparison[tgt_label]
        models = [m for m in MODEL_ORDER if m in results]
        vals = [results[m][metric_key] for m in models]
        colors = [COLORS[m] for m in models]

        bars = ax.bar(range(len(models)),
                      vals,
                      color=colors,
                      edgecolor="white",
                      linewidth=0.8,
                      alpha=0.5,
                      zorder=3)

        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)

        for i, mod in enumerate(models):
            if mod.startswith("pooled_zs"):
                bars[i].set_edgecolor("black")
                bars[i].set_linewidth(1.5)
                bars[i].set_alpha(1.0)

        for bar, val in zip(bars, vals):
            y_pos = bar.get_height() + 0.02 if val >= 0 else bar.get_height(
            ) - 0.08
            ax.text(bar.get_x() + bar.get_width() / 2,
                    y_pos,
                    f"{val:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=9)

        if is_r2:
            ax.axhline(0,
                       color="black",
                       linewidth=0.8,
                       linestyle="--",
                       zorder=2)

        ax.set_xticks(range(len(models)))
        ax.set_xticklabels([LABELS[m] for m in models],
                           rotation=90,
                           ha="right",
                           fontsize=NATURE_SPECS["font_max_pt"])

        if col == 0:
            ax.set_ylabel(metric_label, fontsize=NATURE_SPECS["font_min_pt"])

        valid_vals = [v for v in vals if not np.isnan(v)]
        if is_r2:
            ymin = min(min(valid_vals) * 1.3, -0.2)
            ymax = max(max(valid_vals) * 1.3, 0.5)
        else:
            ymin = 0
            ymax = max(valid_vals) * 1.25
        ax.set_ylim(ymin, ymax)

        n_glaciers = REGION_GLACIER_COUNTS.get(tgt_label)
        title_str = "Target: {}".format(REGION_LABELS.get(
            tgt_label, tgt_label))
        if n_glaciers is not None:
            title_str += f" ({n_glaciers} glaciers)"

        ax.set_title(title_str,
                     fontsize=NATURE_SPECS["font_max_pt"] - 1,
                     pad=6)

# Hide any unused axes in the last row
for empty_idx in range(n_tgts, n_rows_per_metric * n_cols):
    sub_row = empty_idx // n_cols
    col = empty_idx % n_cols
    for metric_row in range(n_metrics):
        axes[metric_row * n_rows_per_metric + sub_row, col].set_visible(False)

# fig.suptitle("Pooled model (zero-shot) vs single-source transformers",
#              fontsize=NATURE_SPECS["font_max_pt"] + 2)
fig.tight_layout(pad=1.8)
fig.subplots_adjust(top=0.9)
fig.savefig("figures/paperTF/fig_zs_pooled_vs_ss_barplot_a.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()

##### Winter:

In [ ]:
# --- bar plot ---
MODEL_ORDER = ["pooled_zs"] + SOURCE_REGIONS
LABELS = {src: f"{src} only" for src in SOURCE_REGIONS}
LABELS["pooled_zs"] = "Pooled"
METRICS = [
    ("RMSE_winter", "RMSE winter (m w.e.)", False),
]

MAX_COLS = 4

EXCLUDE_REGIONS = {'CENTRALASIA', 'ALA'}
tgt_labels = [
    t for t in df_metrics_comparison.keys() if t not in EXCLUDE_REGIONS
]
n_tgts = len(tgt_labels)
n_metrics = len(METRICS)
n_cols = min(MAX_COLS, n_tgts)
n_rows_per_metric = math.ceil(n_tgts / n_cols)
n_rows = n_metrics * n_rows_per_metric

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(3.5 * n_cols, 3.8 * n_rows),
    sharey=False,
)
axes = np.array(axes).reshape(n_rows, n_cols)

for metric_row, (metric_key, metric_label, is_r2) in enumerate(METRICS):
    for tgt_idx, tgt_label in enumerate(tgt_labels):
        sub_row = tgt_idx // n_cols
        col = tgt_idx % n_cols
        row = metric_row * n_rows_per_metric + sub_row
        ax = axes[row, col]

        # subplot label
        ax.text(
            0.03,
            0.97,
            f"({chr(ord('a') + tgt_idx)})",
            transform=ax.transAxes,
            fontsize=NATURE_SPECS["font_max_pt"],
            #fontweight="bold",
            va="top",
            ha="left",
            zorder=5)

        # ax.text(
        # 0.03,
        # 0.97,
        # f"({to_roman(tgt_idx + 1)})",
        # transform=ax.transAxes,
        # fontsize=NATURE_SPECS["font_max_pt"],
        # va="top",
        # ha="left",
        # zorder=5)

        results = df_metrics_comparison[tgt_label]
        models = [m for m in MODEL_ORDER if m in results]
        vals = [results[m][metric_key] for m in models]
        colors = [COLORS[m] for m in models]

        bars = ax.bar(range(len(models)),
                      vals,
                      color=colors,
                      edgecolor="white",
                      linewidth=0.8,
                      alpha=0.5,
                      zorder=3)

        for i, mod in enumerate(models):
            if mod.startswith("pooled_zs"):
                bars[i].set_edgecolor("black")
                bars[i].set_linewidth(1.5)
                bars[i].set_alpha(1.0)

        for bar, val in zip(bars, vals):
            y_pos = bar.get_height() + 0.02 if val >= 0 else bar.get_height(
            ) - 0.08
            ax.text(bar.get_x() + bar.get_width() / 2,
                    y_pos,
                    f"{val:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=8)

        if is_r2:
            ax.axhline(0,
                       color="black",
                       linewidth=0.8,
                       linestyle="--",
                       zorder=2)

        apply_nature_style(ax, fontsize=NATURE_SPECS["font_min_pt"], box=True)

        ax.set_xticks(range(len(models)))
        ax.set_xticklabels([LABELS[m] for m in models],
                           rotation=90,
                           ha="right",
                           fontsize=NATURE_SPECS["font_min_pt"])

        if col == 0:
            ax.set_ylabel(metric_label, fontsize=NATURE_SPECS["font_min_pt"])

        valid_vals = [v for v in vals if not np.isnan(v)]
        if is_r2:
            ymin = min(min(valid_vals) * 1.3, -0.2)
            ymax = max(max(valid_vals) * 1.3, 0.5)
        else:
            ymin = 0
            ymax = max(valid_vals) * 1.25
        ax.set_ylim(ymin, ymax)

        n_glaciers = REGION_GLACIER_COUNTS.get(tgt_label)
        title_str = "Target: {}".format(REGION_LABELS.get(
            tgt_label, tgt_label))
        if n_glaciers is not None:
            title_str += f" ({n_glaciers} glaciers)"

        ax.set_title(title_str,
                     fontsize=NATURE_SPECS["font_max_pt"] - 1,
                     pad=6)

# Hide any unused axes in the last row
for empty_idx in range(n_tgts, n_rows_per_metric * n_cols):
    sub_row = empty_idx // n_cols
    col = empty_idx % n_cols
    for metric_row in range(n_metrics):
        axes[metric_row * n_rows_per_metric + sub_row, col].set_visible(False)

fig.suptitle(
    "Pooled model (zero-shot) vs single-source transformers: winter mass balance",
    fontsize=NATURE_SPECS["font_max_pt"] + 2)
fig.tight_layout(pad=1.8)
fig.subplots_adjust(top=0.9)
fig.savefig("figures/paperTF/appendix/app_zs_pooled_vs_ss_barplot_w.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()

##### Combined:

In [ ]:
METRICS = [
    ("RMSE_annual", "RMSE annual (m w.e.)", False),
    ("RMSE_winter", "RMSE winter (m w.e.)", False),
]

# --- bar plot ---
MODEL_ORDER = ["pooled_zs"] + SOURCE_REGIONS
LABELS = {src: f"{src} only" for src in SOURCE_REGIONS}
LABELS["pooled_zs"] = "Pooled"

MAX_COLS = 4

EXCLUDE_REGIONS = {'CENTRALASIA', 'ALA'}
tgt_labels = [
    t for t in df_metrics_comparison.keys() if t not in EXCLUDE_REGIONS
]
n_tgts = len(tgt_labels)
n_metrics = len(METRICS)
n_cols = min(MAX_COLS, n_tgts)
n_rows_per_metric = math.ceil(n_tgts / n_cols)
n_rows = n_metrics * n_rows_per_metric
n_groups = math.ceil(n_tgts / n_cols)

fig = plt.figure(figsize=(3.5 * n_cols, 3 * n_rows))

# outer gridspec: one row per target group, with extra space between groups
outer_gs = gridspec.GridSpec(
    n_groups,
    1,
    hspace=0.3,
)

# inner gridspec: n_metrics rows per group (annual + winter), tightly packed
axes = np.full((n_groups, n_metrics, n_cols), None, dtype=object)
for g in range(n_groups):
    inner_gs = gridspec.GridSpecFromSubplotSpec(
        n_metrics,
        n_cols,
        subplot_spec=outer_gs[g],
        hspace=0.08,
        wspace=0.3,
    )
    for m in range(n_metrics):
        for c in range(n_cols):
            axes[g, m, c] = fig.add_subplot(inner_gs[m, c])

# ── share y axes within each metric row of each group ────────────────────────
# share y axes within each metric row of each group
for g in range(n_groups):
    for m in range(n_metrics):
        row_axes = [
            axes[g, m, c] for c in range(n_cols) if axes[g, m, c] is not None
        ]
        for ax in row_axes[1:]:
            ax.sharey(row_axes[0])

# ── plot ──────────────────────────────────────────────────────────────────────
for tgt_idx, tgt_label in enumerate(tgt_labels):
    col = tgt_idx % n_cols
    tgt_group = tgt_idx // n_cols

    for metric_row, (metric_key, metric_label, is_r2) in enumerate(METRICS):
        ax = axes[tgt_group, metric_row, col]

        label_idx = tgt_idx + 1 if metric_row == 0 else tgt_idx + 1 + n_tgts
        ax.text(0.03,
                0.97,
                f"({to_roman(label_idx)})",
                transform=ax.transAxes,
                fontsize=NATURE_SPECS["font_max_pt"],
                va="top",
                ha="left",
                zorder=5)

        results = df_metrics_comparison[tgt_label]
        models = [m for m in MODEL_ORDER if m in results]
        vals = [results[m][metric_key] for m in models]
        colors = [COLORS[m] for m in models]

        bars = ax.bar(range(len(models)),
                      vals,
                      color=colors,
                      edgecolor="white",
                      linewidth=0.8,
                      alpha=0.5,
                      zorder=3)

        for i, mod in enumerate(models):
            if mod.startswith("pooled_zs"):
                bars[i].set_edgecolor("black")
                bars[i].set_linewidth(1.5)
                bars[i].set_alpha(1.0)

        for bar, val in zip(bars, vals):
            y_pos = bar.get_height() + 0.02
            ax.text(bar.get_x() + bar.get_width() / 2,
                    y_pos,
                    f"{val:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=8)

        ax.set_xticks(range(len(models)))
        if metric_row == n_metrics - 1:
            ax.set_xticklabels([LABELS[m] for m in models],
                               rotation=90,
                               ha="right",
                               fontsize=NATURE_SPECS["font_min_pt"])
        else:
            ax.set_xticklabels([])
            ax.tick_params(axis="x", length=0)

        if col == 0:
            ax.set_ylabel(metric_label, fontsize=NATURE_SPECS["font_min_pt"])
        else:
            ax.set_ylabel("")
            ax.tick_params(axis="y", labelleft=False)

        if metric_row == 0:
            n_glaciers = REGION_GLACIER_COUNTS.get(tgt_label)
            title_str = "Target: {}".format(
                REGION_LABELS.get(tgt_label, tgt_label))
            if n_glaciers is not None:
                title_str += f" ({n_glaciers} glaciers)"

            ax.set_title(title_str,
                         fontsize=NATURE_SPECS["font_max_pt"] - 1,
                         pad=6)

        apply_nature_style(ax, fontsize=NATURE_SPECS["font_min_pt"], box=True)

# ── set shared y limits per metric row per group, after all bars are drawn ───
for g in range(n_groups):
    for m in range(n_metrics):
        metric_key = METRICS[m][0]
        group_tgts = [tgt_labels[i] for i in range(n_tgts) if i // n_cols == g]
        all_vals = []
        for tgt_label in group_tgts:
            results = df_metrics_comparison[tgt_label]
            all_vals += [
                results[mod][metric_key] for mod in MODEL_ORDER
                if mod in results
            ]
        ymax = max(v for v in all_vals if not np.isnan(v)) * 1.25
        for c in range(n_cols):
            if axes[g, m, c] is not None:
                axes[g, m, c].set_ylim(0, ymax)

# ── hide unused axes in the last group ───────────────────────────────────────
for empty_idx in range(n_tgts, n_groups * n_cols):
    col = empty_idx % n_cols
    tgt_group = empty_idx // n_cols
    for metric_row in range(n_metrics):
        axes[tgt_group, metric_row, col].set_visible(False)

# fig.savefig("figures/paperTF/fig_pooled_zs_vs_single_source_barplot.png",
#             dpi=NATURE_SPECS["dpi"], bbox_inches="tight")
plt.show()

#### Scatter:

In [ ]:
def _eval_and_plot(model,
                   ds_test,
                   scaler,
                   ax,
                   lim,
                   legend_fontsize=NATURE_SPECS["font_min_pt"]):
    ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_test)
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_copy, scaler, batch_size=128, seed=cfg.seed)
    metrics, test_df_preds = model.evaluate_with_preds(device, test_dl,
                                                       ds_copy)
    test_df_preds = test_df_preds.dropna(subset=["target", "pred"])
    scores_annual, scores_winter = compute_seasonal_scores(test_df_preds,
                                                           target_col="target",
                                                           pred_col="pred")

    pred_vs_truth_density(
        ax,
        test_df_preds,
        scores_annual,
        add_legend=False,
        palette=[mbm.plots.COLOR_ANNUAL, mbm.plots.COLOR_WINTER],
        ax_xlim=lim,
        ax_ylim=lim,
        s=50,
    )

    def _fmt(x):
        return "NA" if (x is None or
                        (isinstance(x, float) and np.isnan(x))) else f"{x:.2f}"

    legend_NN = "\n".join([
        rf"$\mathrm{{RMSE_a}}={_fmt(scores_annual['rmse'])},\ \mathrm{{RMSE_w}}={_fmt(scores_winter['rmse'])}$",
        rf"$\mathrm{{R^2_a}}={_fmt(scores_annual['R2'])},\ \mathrm{{R^2_w}}={_fmt(scores_winter['R2'])}$",
        rf"$\mathrm{{Bias_a}}={_fmt(scores_annual['Bias'])},\ \mathrm{{Bias_w}}={_fmt(scores_winter['Bias'])}$",
    ])
    ax.text(0.98,
            0.02,
            legend_NN,
            transform=ax.transAxes,
            va="bottom",
            ha="right",
            fontsize=legend_fontsize,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.5))
    return metrics


EXCLUDE_REGIONS = {'CENTRALASIA', 'ALA'}
regions_to_plot = [
    t for t in df_metrics_comparison.keys() if t not in EXCLUDE_REGIONS
]

plot_model_order = ["pooled_zs"] + SOURCE_REGIONS
COL_TITLES_ZS = {
    "pooled_zs": "Pooled (zero-shot)",
    **{
        src: f"Source: {REGION_LABELS.get(src, src)}"
        for src in SOURCE_REGIONS
    },
}
model_and_scaler = {
    "pooled_zs": (model_pooled, ds_pooled_scaler),
    **{
        src: (single_source_models[src], single_source_scalers[src])
        for src in SOURCE_REGIONS
    },
}

ncols = len(plot_model_order)
nrows = len(regions_to_plot)

region_lims_zs = {}
for region in regions_to_plot:
    ds_test = dict(eval_configs)[region]["ds_test"]
    vals = ds_test.y.cpu().numpy().flatten()
    vals = vals[~np.isnan(vals)]
    pad = (vals.max() - vals.min()) * 0.5
    region_lims_zs[region] = (float(vals.min() - pad), float(vals.max() + pad))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(4 * ncols, 4 * nrows),
    sharex="row",
    sharey="row",
)
axes = np.array(axes).reshape(nrows, ncols)

eval_configs_dict = dict(eval_configs)

for row_i, region in enumerate(regions_to_plot):
    lim = region_lims_zs[region]
    ds_test = eval_configs_dict[region]["ds_test"]

    for col_i, model_key in enumerate(plot_model_order):
        ax = axes[row_i, col_i]
        model, scaler = model_and_scaler[model_key]

        _eval_and_plot(model,
                       ds_test,
                       scaler,
                       ax,
                       lim,
                       legend_fontsize=NATURE_SPECS["font_min_pt"])
        apply_nature_style(ax,
                           fontsize=NATURE_SPECS["font_max_pt"] + 2,
                           box=True)
        leg = ax.get_legend()
        if leg:
            leg.remove()

        row_letter = chr(ord('a') + row_i)
        panel_label = f"{row_letter}{col_i + 1}"
        ax.text(
            0.02,
            0.98,
            f"({panel_label})",
            transform=ax.transAxes,
            fontsize=NATURE_SPECS["font_max_pt"],
            #fontweight="bold",
            va="top",
            ha="left")

        if row_i == 0:
            ax.set_title(
                COL_TITLES_ZS.get(model_key, model_key),
                fontsize=NATURE_SPECS["font_max_pt"] + 5,
            )
        if col_i == 0:
            ax.set_ylabel(
                f"{REGION_LABELS.get(region, region)}\nModeled PMB (m w.e.)",
                fontsize=NATURE_SPECS["font_max_pt"] + 2)
        else:
            ax.set_ylabel("")
        if row_i < nrows - 1:
            ax.set_xlabel("")

fig.legend(
    handles=[
        mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
        mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncols=2,
    frameon=False,
    fontsize=NATURE_SPECS["font_max_pt"] + 5,
)
fig.suptitle(
    "Zero-shot: pooled model vs single-source transformers",
    fontsize=NATURE_SPECS["font_max_pt"] + 6,
    #fontweight="bold",
)
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.subplots_adjust(top=0.96)
fig.savefig("figures/paperTF/appendix/app_zs_pooled_vs_ss_scatter_all.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()

##### Examples:

In [ ]:
COL_TITLES_ZS = {
    "pooled_zs": "Source: Pooled",
    **{
        src: f"Source: {REGION_LABELS.get(src, src)}"
        for src in SOURCE_REGIONS
    },
}

# ── Override: only Italy and Austria as targets ──────────────────────────────
regions_to_plot = ["CA_4", "ALA_6"]

# Recompute axis limits for the two selected regions only
region_lims_zs = {}
for region in regions_to_plot:
    ds_test = dict(eval_configs)[region]["ds_test"]
    vals = ds_test.y.cpu().numpy().flatten()
    vals = vals[~np.isnan(vals)]
    pad = (vals.max() - vals.min()) * 0.5
    region_lims_zs[region] = (float(vals.min() - pad), float(vals.max() + pad))

ncols = len(plot_model_order)  # unchanged: pooled + 4 sources = 5
nrows = len(regions_to_plot)  # now 2

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(4 * ncols, 4 * nrows),  # → (20, 8) instead of (20, 32)
    sharex="row",
    sharey="row",
)
axes = np.array(axes).reshape(nrows, ncols)

eval_configs_dict = dict(eval_configs)

for row_i, region in enumerate(regions_to_plot):
    lim = region_lims_zs[region]
    ds_test = eval_configs_dict[region]["ds_test"]

    for col_i, model_key in enumerate(plot_model_order):
        ax = axes[row_i, col_i]
        model, scaler = model_and_scaler[model_key]

        _eval_and_plot(model,
                       ds_test,
                       scaler,
                       ax,
                       lim,
                       legend_fontsize=NATURE_SPECS["font_min_pt"])
        apply_nature_style(ax,
                           fontsize=NATURE_SPECS["font_max_pt"] + 2,
                           box=True)
        leg = ax.get_legend()
        if leg:
            leg.remove()

        #panel_label = chr(ord('i') + row_i * ncols + col_i)
        panel_label = to_roman(row_i * ncols + col_i + 1 + 8)
        ax.text(0.02,
                0.98,
                f"({panel_label})",
                transform=ax.transAxes,
                fontsize=NATURE_SPECS["font_max_pt"] + 6,
                va="top",
                ha="left")

        if row_i == 0:
            ax.set_title(
                COL_TITLES_ZS.get(model_key, model_key),
                fontsize=NATURE_SPECS["font_max_pt"] + 8,
            )
        if col_i == 0:
            ax.set_ylabel(
                #f"Target: {REGION_LABELS.get(region, region)}\nModeled PMB (m w.e.)",
                "Modeled PMB (m w.e.)",
                fontsize=NATURE_SPECS["font_max_pt"] + 2)
        else:
            ax.set_ylabel("")
        if row_i < nrows - 1:
            ax.set_xlabel("")

fig.legend(
    handles=[
        mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
        mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncols=2,
    frameon=False,
    fontsize=NATURE_SPECS["font_max_pt"] + 4,
)
# fig.suptitle(
#     "Zero-shot: pooled model vs single-source transformers",
#     fontsize=NATURE_SPECS["font_max_pt"] + 2,
# )
fig.tight_layout(rect=[0, 0.08, 1, 1])
fig.subplots_adjust(top=0.93)
fig.savefig(
    "figures/paperTF/fig_zs_pooled_vs_ss_scatter_examples.png",
    dpi=NATURE_SPECS["dpi"],
    bbox_inches="tight",
)
plt.show()

### Fine tuning:

#### Simple fine tuning:

In [ ]:
models_dir_ft = BASE_DIR / "models" / "finetuned"
os.makedirs(models_dir_ft, exist_ok=True)

ft_models_gl = {}  # glacier-split
ft_models_id = {}  # ID-split, now nested by rep

# ── glacier-split: unchanged structure ────────────────────────────────────────
for region in gl_level_split:
    ft_models_gl[region] = {}
    print(f"\n{'='*60}")
    print(f"Fine-tuning on: {region} (GLsplit)")

    for strategy in ["head_only", "full", "adapter"]:
        ft_models_gl[region][strategy] = finetune_transformer_on_target(
            cfg=cfg,
            model_pooled=model_pooled,
            ds_ft=ft_assets_glacier[region]["ds_ft"],
            ds_pooled_scaler=ds_pooled_scaler,
            device=device,
            best_params=PARAMS_TRANSFORMER,
            model_filename=str(
                models_dir_ft /
                f"transformer_ft_{strategy}_{region}_{MODEL_DATE}.pt"),
            strategy=strategy,
            epochs=PARAMS_FT['epochs'],
            lr_factor=PARAMS_FT['lr_factor'],
            force_retrain=False,
        )
        print(f"  [{region}] {strategy} done")

# ── ID-split: loop over reps, one model set per rep ──────────────────────────
for region in id_level_split:
    ft_models_id[region] = []  # list of per-rep strategy dicts
    print(f"\n{'='*60}")
    print(f"Fine-tuning on: {region} (IDsplit, {N_REPEATS} repeats)")

    for rep_data in ft_assets_id[region]:
        rep = rep_data["rep"]
        rep_models = {}

        for strategy in ["head_only", "full", "adapter"]:
            rep_models[strategy] = finetune_transformer_on_target(
                cfg=cfg,
                model_pooled=model_pooled,
                ds_ft=rep_data["ds_ft"],
                ds_pooled_scaler=ds_pooled_scaler,
                device=device,
                best_params=PARAMS_TRANSFORMER,
                model_filename=str(
                    models_dir_ft /
                    f"transformer_ft_{strategy}_{region}_IDsplit_rep{rep}_{MODEL_DATE}.pt"
                ),
                strategy=strategy,
                epochs=PARAMS_FT['epochs'],
                lr_factor=PARAMS_FT['lr_factor'],
                force_retrain=False,
            )
            print(f"  [{region}] rep {rep} - {strategy} done")

        ft_models_id[region].append({"rep": rep, "models": rep_models})

#### DAN fine tuning:

In [ ]:
source_codes_pretrain = build_source_codes_for_dataset(
    assets_full["ds_train"],
    pd.concat(
        [res_xreg_by_source[r]["df_train"] for r in SOURCE_REGIONS],
        ignore_index=True,
    ),
    source_col="SOURCE_CODE",
)

dan_models = {}  # glacier-split
dan_models_id = {}  # ID-split, now nested by rep

for region in id_level_split:
    print(f"\n{'='*50}  DAN → {region} (IDsplit, {N_REPEATS} repeats)")

    dan_models_id[region] = []  # list of per-rep DAN models

    for rep_data in ft_assets_id[region]:
        rep = rep_data["rep"]

        source_codes_ft = [region] * len(rep_data["ds_ft"])

        dan_model, _ = finetune_dan_transformer_on_target(
            cfg=cfg,
            model_foundation=model_pooled,
            assets_full=assets_full,
            ft_assets_region=rep_data,
            ds_pooled_scaler=ds_pooled_scaler,
            source_codes_pretrain=source_codes_pretrain,
            source_codes_ft=source_codes_ft,
            device=device,
            best_params=PARAMS_TRANSFORMER,
            model_filename=str(
                models_dir_ft /
                f"transformer_dan_{region}_IDsplit_rep{rep}_{MODEL_DATE}.pt"),
            dan_alpha=PARAMS_DAN['dan_alpha'],
            grl_lambda=PARAMS_DAN['grl_lambda'],
            mix_ratio_ft=PARAMS_DAN['mix_ratio_ft'],
            epochs=PARAMS_DAN['epochs'],
            lr_backbone=PARAMS_DAN['lr_backbone'],
            lr_domain=PARAMS_DAN['lr_domain'],
            force_retrain=False,
            verbose=False,
        )

        dan_models_id[region].append({"rep": rep, "model": dan_model})
        print(f"  [{region}] rep {rep} DAN done")

In [ ]:
# --- unified evaluation --- ID-split only, aggregated over repeats
print(f"\n{'='*75}")
print(f"{'Strategy':<25} {'RMSE_a':>8} {'RMSE_w':>8} {'R2_a':>8} {'R2_w':>8}")
print(f"{'='*75}")

df_ft_metrics_id = {}  # {region: {strategy_label: {metric: mean_value}}}
df_ft_metrics_id_raw = {
}  # {region: {strategy_label: [metrics_per_rep]}} for std/error bars

for region in id_level_split:
    print(f"\n--- {region} (mean over {N_REPEATS} repeats) ---")

    df_ft_metrics_id_raw[region] = {
        label: []
        for label in ["no_ft", "head_only", "adapter", "full", "dan"]
    }

    for rep_idx, rep_data in enumerate(ft_assets_id[region]):
        rep = rep_data["rep"]

        def _eval(model, ds_test=rep_data["ds_test"]):
            ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                ds_test)
            test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
                ds_test_copy, ds_pooled_scaler, batch_size=128, seed=cfg.seed)
            return model.evaluate_with_preds(device, test_dl, ds_test_copy)

        # models for this rep
        rep_ft_models = ft_models_id[region][rep_idx]["models"]
        rep_dan_model = dan_models_id[region][rep_idx]["model"]

        all_models = {
            "no_ft": model_pooled,
            "head_only": rep_ft_models["head_only"],
            "adapter": rep_ft_models["adapter"],
            "full": rep_ft_models["full"],
            "dan": rep_dan_model,
        }

        for label, model in all_models.items():
            m, _ = _eval(model)
            df_ft_metrics_id_raw[region][label].append(m)

    # aggregate across reps
    df_ft_metrics_id[region] = {}
    for label, rep_metrics_list in df_ft_metrics_id_raw[region].items():
        agg = {}
        for metric_key in rep_metrics_list[0].keys():
            vals = [m[metric_key] for m in rep_metrics_list]
            agg[metric_key] = float(np.mean(vals))
            agg[f"{metric_key}_std"] = float(np.std(vals))
        df_ft_metrics_id[region][label] = agg

        print(f"  {label:<25}  "
              f"RMSE_a={agg['RMSE_annual']:.3f}±{agg['RMSE_annual_std']:.3f}  "
              f"RMSE_w={agg['RMSE_winter']:.3f}±{agg['RMSE_winter_std']:.3f}  "
              f"R2_a={agg['R2_annual']:.3f}±{agg['R2_annual_std']:.3f}  "
              f"R2_w={agg['R2_winter']:.3f}±{agg['R2_winter_std']:.3f}")

#### Scatter plot:

In [ ]:
COL_TITLES = {
    "no_ft (zero-shot)": "Zero-shot",
    "full": "Full FT",
    "adapter": "Adapter FT",
    "dan": "DAN",
}
INCLUDE_REGIONS = ["IT", 'SJM', 'CA_3', 'ALA_4']  # adjust for paper figure
regions_to_plot = [r for r in id_level_split if r in INCLUDE_REGIONS]

strategies = ["full"]

plot_configs_labels = ["no_ft (zero-shot)"] + strategies + ["dan"]
ncols = len(plot_configs_labels)
nrows = len(regions_to_plot)

PLOT_REP = 0  # which repeat to visualise in this scatter figure

# Precompute per-region axis limits, using the chosen rep's test set
region_lims = {}
for region in regions_to_plot:
    rep_data = ft_assets_id[region][PLOT_REP]
    vals = rep_data["ds_test"].y.cpu().numpy().flatten()
    vals = vals[~np.isnan(vals)]
    pad = (vals.max() - vals.min()) * 0.3
    lim = (float(vals.min() - pad), float(vals.max() + pad))
    region_lims[region] = lim

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(4 * ncols, 4 * nrows),
    sharex="row",
    sharey="row",
)
axes = np.array(axes).reshape(nrows, ncols)

for row_i, region in enumerate(regions_to_plot):
    lim = region_lims[region]
    rep_data = ft_assets_id[region][PLOT_REP]

    # pull the rep-matched models for this region
    rep_ft_models = next(d["models"] for d in ft_models_id[region]
                         if d["rep"] == PLOT_REP)
    rep_dan_model = next(d["model"] for d in dan_models_id[region]
                         if d["rep"] == PLOT_REP)

    plot_configs = [
        ("no_ft (zero-shot)", model_pooled, assets_full),
        *[(strategy, rep_ft_models[strategy], assets_full)
          for strategy in strategies],
        ("dan", rep_dan_model, assets_full),
    ]
    for col_i, (label, model, assets_train) in enumerate(plot_configs):
        ax = axes[row_i, col_i]
        evaluate_one_model(
            cfg=cfg,
            model=model,
            device=device,
            lstm_assets_for_key={
                "ds_train": assets_train["ds_train"],
                "ds_test": rep_data["ds_test"],
                "train_idx": assets_train["train_idx"],
                "val_idx": assets_train["val_idx"],
            },
            ax=ax,
            ax_xlim=lim,
            ax_ylim=lim,
        )
        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)
        leg = ax.get_legend()
        if leg:
            leg.remove()

        row_letter = chr(ord('a') + row_i)
        panel_label = f"{row_letter}{col_i + 1}"
        ax.text(0.02,
                0.98,
                f"({panel_label})",
                transform=ax.transAxes,
                fontsize=NATURE_SPECS["font_max_pt"],
                va="top",
                ha="left")

        if row_i == 0:
            ax.set_title(
                COL_TITLES.get(label, label),
                fontsize=NATURE_SPECS["font_max_pt"] + 1,
            )
        if col_i == 0:
            ax.set_ylabel(
                f"{REGION_LABELS.get(region, region)}\nModeled PMB (m w.e.)",
                fontsize=NATURE_SPECS["font_max_pt"])
        else:
            ax.set_ylabel("")
        if row_i < nrows - 1:
            ax.set_xlabel("")

fig.legend(
    handles=[
        mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
        mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.0),
    ncols=2,
    frameon=False,
    fontsize=NATURE_SPECS["font_max_pt"] + 2,
)
fig.suptitle(
    f"Pooled model: zero-shot vs fine-tuning strategies (ID-split, repeat {PLOT_REP})",
    fontsize=NATURE_SPECS["font_max_pt"] + 2,
    y=1.01,
)
fig.tight_layout(rect=[0, 0.04, 1, 0.99])
plt.show()

In [ ]:
import matplotlib.gridspec as gridspec

EXCLUDE_REGIONS = {'CENTRALASIA', 'ALA'}
regions_to_plot = [r for r in id_level_split if r not in EXCLUDE_REGIONS]
strategies = ["full"]

PLOT_REP = 0  # which repeat to visualise in this appendix figure

COL_TITLES = {
    "no_ft (zero-shot)": "Zero-shot",
    "full": "Full FT",
    "adapter": "Adapter FT",
    "dan": "DAN",
}

plot_configs_labels = ["no_ft (zero-shot)"] + strategies + ["dan"]
nrows_per_group = len(plot_configs_labels)
MAX_COLS = 4
n_regions = len(regions_to_plot)
n_cols = min(MAX_COLS, n_regions)
n_groups = math.ceil(n_regions / n_cols)

# Precompute per-region axis limits, using the chosen rep's test set
region_lims = {}
for region in regions_to_plot:
    rep_data = ft_assets_id[region][PLOT_REP]
    vals = rep_data["ds_test"].y.cpu().numpy().flatten()
    vals = vals[~np.isnan(vals)]
    pad = (vals.max() - vals.min()) * 0.5
    region_lims[region] = (float(vals.min() - pad), float(vals.max() + pad))

fig = plt.figure(figsize=(4 * n_cols, 3.4 * nrows_per_group * n_groups))

outer_gs = gridspec.GridSpec(
    n_groups,
    1,
    hspace=0.2,
    top=0.94,
    bottom=0.06,
)
axes = np.full((n_groups, nrows_per_group, n_cols), None, dtype=object)
for g in range(n_groups):
    inner_gs = gridspec.GridSpecFromSubplotSpec(
        nrows_per_group,
        n_cols,
        subplot_spec=outer_gs[g],
        hspace=0.15,
        wspace=0.25,
    )
    for r in range(nrows_per_group):
        for c in range(n_cols):
            axes[g, r, c] = fig.add_subplot(inner_gs[r, c])

# share y within each strategy row of each group (sharex similarly if desired)
for g in range(n_groups):
    for r in range(nrows_per_group):
        row_axes = [
            axes[g, r, c] for c in range(n_cols) if axes[g, r, c] is not None
        ]
        for ax in row_axes[1:]:
            ax.sharey(row_axes[0])
            ax.sharex(row_axes[0])

for region_idx, region in enumerate(regions_to_plot):
    col = region_idx % n_cols
    group = region_idx // n_cols
    lim = region_lims[region]

    rep_data = ft_assets_id[region][PLOT_REP]
    rep_ft_models = next(d["models"] for d in ft_models_id[region]
                         if d["rep"] == PLOT_REP)
    rep_dan_model = next(d["model"] for d in dan_models_id[region]
                         if d["rep"] == PLOT_REP)

    plot_configs = [
        ("no_ft (zero-shot)", model_pooled, assets_full),
        *[(strategy, rep_ft_models[strategy], assets_full)
          for strategy in strategies],
        ("dan", rep_dan_model, assets_full),
    ]

    for row_i, (label, model, assets_train) in enumerate(plot_configs):
        ax = axes[group, row_i, col]
        evaluate_one_model(
            cfg=cfg,
            model=model,
            device=device,
            lstm_assets_for_key={
                "ds_train": assets_train["ds_train"],
                "ds_test": rep_data["ds_test"],
                "train_idx": assets_train["train_idx"],
                "val_idx": assets_train["val_idx"],
            },
            ax=ax,
            ax_xlim=lim,
            ax_ylim=lim,
        )
        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)
        leg = ax.get_legend()
        if leg:
            leg.remove()

        panel_label = chr(ord('a') + region_idx) + str(row_i + 1)
        ax.text(0.02,
                0.98,
                f"({panel_label})",
                transform=ax.transAxes,
                fontsize=NATURE_SPECS["font_max_pt"],
                va="top",
                ha="left")

        if row_i == 0:
            ax.set_title(f"Target: {REGION_LABELS.get(region, region)}",
                         fontsize=NATURE_SPECS["font_max_pt"] + 1,
                         fontweight='bold')

        if col == 0:
            ax.set_ylabel(
                f"{COL_TITLES.get(label, label)}\nModeled PMB (m w.e.)",
                fontsize=NATURE_SPECS["font_max_pt"])
        else:
            ax.set_ylabel("")
            ax.tick_params(axis="y", labelleft=False)

        if row_i < nrows_per_group - 1:
            ax.set_xlabel("")
            ax.tick_params(axis="x", labelbottom=False)

# hide unused axes if n_regions isn't a multiple of n_cols
for empty_idx in range(n_regions, n_groups * n_cols):
    col = empty_idx % n_cols
    group = empty_idx // n_cols
    for row_i in range(nrows_per_group):
        axes[group, row_i, col].set_visible(False)

fig.legend(
    handles=[
        mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
        mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.005),
    ncols=2,
    frameon=False,
    fontsize=NATURE_SPECS["font_max_pt"],
)
fig.suptitle(
    f"Pooled model: zero-shot vs fine-tuning strategies (ID-split, repeat {PLOT_REP})",
    fontsize=NATURE_SPECS["font_max_pt"] + 2,
)
fig.savefig("figures/paperTF/appendix/app_ft_strategies_all.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()

#### Bar plot:

##### Annual:

In [ ]:
# ── Fine-tuning gain barplot ──────────────────────────────────────────────────

FT_MODEL_ORDER = ["no_ft", "head_only", "full", "dan"]
FT_LABELS = {
    "no_ft": "Zero-shot",
    "head_only": "Head only",
    "full": "Full FT",
    "dan": "DAN",
}
FT_COLORS = {
    "no_ft": "#aaaaaa",
    "head_only": "#EE854A",
    "full": "#D65F5F",
    "dan": "#956CB4",
}

METRICS_FT = [
    ("RMSE_annual", "RMSE annual (m w.e.)", False),
]

MAX_COLS_FT = 4
EXCLUDE_FT = {"CENTRALASIA", "ALA"}

tgt_labels_ft = [t for t in df_ft_metrics_id.keys() if t not in EXCLUDE_FT]

n_tgts_ft = len(tgt_labels_ft)
n_metrics_ft = len(METRICS_FT)
n_cols_ft = min(MAX_COLS_FT, n_tgts_ft)
n_col_groups = math.ceil(n_tgts_ft / n_cols_ft)

PLOT_H = 2.5
SPACER_H = 1.3

height_ratios = []
for g in range(n_col_groups):
    height_ratios.append(PLOT_H)
    if g < n_col_groups - 1:
        height_ratios.append(SPACER_H)

total_rows = len(height_ratios)
fig_h = sum(height_ratios) + 0.8

fig = plt.figure(figsize=(3.5 * n_cols_ft, fig_h))
gs = GridSpec(
    total_rows,
    n_cols_ft,
    figure=fig,
    height_ratios=height_ratios,
    hspace=0.18,
    wspace=0.35,
)


def gs_row(group, metric_offset):
    return group * 2


# ── pre-compute shared ylims per (group_row, metric_offset), accounting for error bars
shared_ylims = {}
for tgt_idx, tgt_label in enumerate(tgt_labels_ft):
    group_row = tgt_idx // n_cols_ft
    results = df_ft_metrics_id[tgt_label]
    models = [m for m in FT_MODEL_ORDER if m in results]

    for metric_offset, (metric_key, _, is_r2) in enumerate(METRICS_FT):
        std_key = f"{metric_key}_std"
        vals = [results[m][metric_key] for m in models]
        stds = [results[m].get(std_key, 0.0) for m in models]
        valid_pairs = [(v, s) for v, s in zip(vals, stds) if not np.isnan(v)]
        if not valid_pairs:
            continue
        valid_vals = [v for v, _ in valid_pairs]
        valid_tops = [v + s
                      for v, s in valid_pairs]  # account for error bar extent
        if is_r2:
            ymin = min(min(valid_vals) * 1.3, -0.2)
            ymax = max(max(valid_tops) * 1.3, 0.5)
        else:
            ymin = 0
            ymax = max(valid_tops) * 1.25
        key = (group_row, metric_offset)
        if key not in shared_ylims:
            shared_ylims[key] = (ymin, ymax)
        else:
            shared_ylims[key] = (
                min(shared_ylims[key][0], ymin),
                max(shared_ylims[key][1], ymax),
            )

# ── plot ──────────────────────────────────────────────────────────────────────
for tgt_idx, tgt_label in enumerate(tgt_labels_ft):
    col = tgt_idx % n_cols_ft
    group_row = tgt_idx // n_cols_ft

    results = df_ft_metrics_id[tgt_label]
    models = [m for m in FT_MODEL_ORDER if m in results]
    colors = [FT_COLORS[m] for m in models]

    for metric_offset, (metric_key, metric_label,
                        is_r2) in enumerate(METRICS_FT):
        std_key = f"{metric_key}_std"
        row = gs_row(group_row, metric_offset)
        ax = fig.add_subplot(gs[row, col])

        panel_idx = group_row * n_cols_ft * n_metrics_ft + metric_offset * n_cols_ft + col
        ax.text(0.03,
                0.97,
                f"({chr(ord('a') + panel_idx)})",
                transform=ax.transAxes,
                fontsize=NATURE_SPECS["font_max_pt"],
                va="top",
                ha="left",
                zorder=5)

        vals = [results[m][metric_key] for m in models]
        stds = [results[m].get(std_key, 0.0) for m in models]

        bars = ax.bar(
            range(len(models)),
            vals,
            yerr=stds,
            capsize=3,
            error_kw={
                "linewidth": 1.0,
                "ecolor": "black",
                "zorder": 4
            },
            color=colors,
            edgecolor="white",
            linewidth=0.8,
            alpha=BARPLOT_ALPHA,
            zorder=3,
        )

        for i, mod in enumerate(models):
            if mod == "no_ft":
                bars[i].set_edgecolor("black")
                bars[i].set_linewidth(1.5)
                bars[i].set_alpha(1.0)

        for bar, val, std in zip(bars, vals, stds):
            if np.isnan(val):
                continue
            y_pos = bar.get_height(
            ) + std + 0.02 if val >= 0 else bar.get_height() - std - 0.08
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                y_pos,
                f"{val:.2f}",
                ha="center",
                va="bottom",
                fontsize=9,
            )

        if is_r2:
            ax.axhline(0,
                       color="black",
                       linewidth=0.8,
                       linestyle="--",
                       zorder=2)

        ax.set_xticks(range(len(models)))
        if metric_offset == n_metrics_ft - 1:
            ax.set_xticklabels(
                [FT_LABELS[m] for m in models],
                rotation=90,
                ha="right",
                fontsize=NATURE_SPECS["font_max_pt"],
            )
        else:
            ax.set_xticklabels([])

        if col == 0:
            ax.set_ylabel(metric_label, fontsize=NATURE_SPECS["font_max_pt"])

        if metric_offset == 0:
            ax.set_title(
                REGION_LABELS.get(tgt_label, tgt_label),
                fontsize=NATURE_SPECS["font_max_pt"] + 1,
                fontweight="bold",
                pad=6,
            )

        key = (group_row, metric_offset)
        if key in shared_ylims:
            ax.set_ylim(shared_ylims[key])

        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=False)

fig.suptitle(
    "Foundation model: zero-shot vs fine-tuning strategies (10% FT data, mean ± std over repeats)",
    fontsize=NATURE_SPECS["font_max_pt"] + 2,
    y=1.0,
)
fig.savefig(
    "figures/paperTF/fig_pooled_ft_gain_barplot.png",
    dpi=NATURE_SPECS["dpi"],
    bbox_inches="tight",
)
plt.show()

##### Winter:

In [ ]:
# ── Fine-tuning gain barplot ──────────────────────────────────────────────────
FT_MODEL_ORDER = ["no_ft", "head_only", "full", "dan"]
FT_LABELS = {
    "no_ft": "Zero-shot",
    "head_only": "Head only",
    "full": "Full FT",
    "dan": "DAN",
}
FT_COLORS = {
    "no_ft": "#aaaaaa",
    "head_only": "#EE854A",
    "full": "#D65F5F",
    "dan": "#956CB4",
}

METRICS_FT = [
    ("RMSE_winter", "RMSE winter (m w.e.)", False),
]

MAX_COLS_FT = 4
EXCLUDE_FT = {"CENTRALASIA", "ALA"}

tgt_labels_ft = [t for t in df_ft_metrics_id.keys() if t not in EXCLUDE_FT]

n_tgts_ft = len(tgt_labels_ft)
n_metrics_ft = len(METRICS_FT)
n_cols_ft = min(MAX_COLS_FT, n_tgts_ft)
n_col_groups = math.ceil(n_tgts_ft / n_cols_ft)

PLOT_H = 2.5
SPACER_H = 1.3

height_ratios = []
for g in range(n_col_groups):
    height_ratios.append(PLOT_H)
    if g < n_col_groups - 1:
        height_ratios.append(SPACER_H)

total_rows = len(height_ratios)
fig_h = sum(height_ratios) + 0.8

fig = plt.figure(figsize=(3.5 * n_cols_ft, fig_h))
gs = GridSpec(
    total_rows,
    n_cols_ft,
    figure=fig,
    height_ratios=height_ratios,
    hspace=0.18,
    wspace=0.35,
)


def gs_row(group, metric_offset):
    return group * 2


# ── pre-compute shared ylims per (group_row, metric_offset), accounting for error bars
shared_ylims = {}
for tgt_idx, tgt_label in enumerate(tgt_labels_ft):
    group_row = tgt_idx // n_cols_ft
    results = df_ft_metrics_id[tgt_label]
    models = [m for m in FT_MODEL_ORDER if m in results]

    for metric_offset, (metric_key, _, is_r2) in enumerate(METRICS_FT):
        std_key = f"{metric_key}_std"
        vals = [results[m][metric_key] for m in models]
        stds = [results[m].get(std_key, 0.0) for m in models]
        valid_pairs = [(v, s) for v, s in zip(vals, stds) if not np.isnan(v)]
        if not valid_pairs:
            continue
        valid_vals = [v for v, _ in valid_pairs]
        valid_tops = [v + s for v, s in valid_pairs]
        if is_r2:
            ymin = min(min(valid_vals) * 1.3, -0.2)
            ymax = max(max(valid_tops) * 1.3, 0.5)
        else:
            ymin = 0
            ymax = max(valid_tops) * 1.25
        key = (group_row, metric_offset)
        if key not in shared_ylims:
            shared_ylims[key] = (ymin, ymax)
        else:
            shared_ylims[key] = (
                min(shared_ylims[key][0], ymin),
                max(shared_ylims[key][1], ymax),
            )

# ── plot ──────────────────────────────────────────────────────────────────────
for tgt_idx, tgt_label in enumerate(tgt_labels_ft):
    col = tgt_idx % n_cols_ft
    group_row = tgt_idx // n_cols_ft

    results = df_ft_metrics_id[tgt_label]
    models = [m for m in FT_MODEL_ORDER if m in results]
    colors = [FT_COLORS[m] for m in models]

    for metric_offset, (metric_key, metric_label,
                        is_r2) in enumerate(METRICS_FT):
        std_key = f"{metric_key}_std"
        row = gs_row(group_row, metric_offset)
        ax = fig.add_subplot(gs[row, col])

        panel_idx = group_row * n_cols_ft * n_metrics_ft + metric_offset * n_cols_ft + col
        ax.text(0.03,
                0.97,
                f"({chr(ord('a') + panel_idx)})",
                transform=ax.transAxes,
                fontsize=NATURE_SPECS["font_max_pt"],
                va="top",
                ha="left",
                zorder=5)

        vals = [results[m][metric_key] for m in models]
        stds = [results[m].get(std_key, 0.0) for m in models]

        bars = ax.bar(
            range(len(models)),
            vals,
            yerr=stds,
            capsize=3,
            error_kw={
                "linewidth": 1.0,
                "ecolor": "black",
                "zorder": 4
            },
            color=colors,
            edgecolor="white",
            linewidth=0.8,
            alpha=BARPLOT_ALPHA,
            zorder=3,
        )

        for i, mod in enumerate(models):
            if mod == "no_ft":
                bars[i].set_edgecolor("black")
                bars[i].set_linewidth(1.5)
                bars[i].set_alpha(1.0)

        for bar, val, std in zip(bars, vals, stds):
            if np.isnan(val):
                continue
            y_pos = bar.get_height(
            ) + std + 0.02 if val >= 0 else bar.get_height() - std - 0.08
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                y_pos,
                f"{val:.2f}",
                ha="center",
                va="bottom",
                fontsize=9,
            )

        if is_r2:
            ax.axhline(0,
                       color="black",
                       linewidth=0.8,
                       linestyle="--",
                       zorder=2)

        ax.set_xticks(range(len(models)))
        if metric_offset == n_metrics_ft - 1:
            ax.set_xticklabels(
                [FT_LABELS[m] for m in models],
                rotation=90,
                ha="right",
                fontsize=NATURE_SPECS["font_max_pt"],
            )
        else:
            ax.set_xticklabels([])

        if col == 0:
            ax.set_ylabel(metric_label, fontsize=NATURE_SPECS["font_max_pt"])

        if metric_offset == 0:
            ax.set_title(
                REGION_LABELS.get(tgt_label, tgt_label),
                fontsize=NATURE_SPECS["font_max_pt"] + 1,
                fontweight="bold",
                pad=6,
            )

        key = (group_row, metric_offset)
        if key in shared_ylims:
            ax.set_ylim(shared_ylims[key])

        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=False)

fig.suptitle(
    "Foundation model: zero-shot vs fine-tuning strategies (10% FT data, mean ± std over repeats) - winter mass balance",
    fontsize=NATURE_SPECS["font_max_pt"] + 2,
    y=1.0,
)
fig.savefig(
    "figures/paperTF/appendix/app_pooled_ft_gain_barplot_w.png",
    dpi=NATURE_SPECS["dpi"],
    bbox_inches="tight",
)
plt.show()

##### Combined:

In [ ]:
# ── Fine-tuning gain barplot ──────────────────────────────────────────────────

FT_MODEL_ORDER = ["no_ft", "head_only", "full", "dan"]
FT_LABELS = {
    "no_ft": "Zero-shot",
    "head_only": "Head only",
    "full": "Full FT",
    "dan": "DAN",
}
FT_COLORS = {
    "no_ft": "#aaaaaa",
    "head_only": "#EE854A",
    "full": "#D65F5F",
    "dan": "#956CB4",
}

METRICS_FT = [
    ("RMSE_annual", "RMSE annual (m w.e.)", False),
    ("RMSE_winter", "RMSE winter (m w.e.)", False),
]

MAX_COLS_FT = 4
EXCLUDE_FT = {"CENTRALASIA", "ALA"}

tgt_labels_ft = [t for t in df_ft_metrics_id.keys() if t not in EXCLUDE_FT]

n_tgts_ft = len(tgt_labels_ft)
n_metrics_ft = len(METRICS_FT)
n_cols_ft = min(MAX_COLS_FT, n_tgts_ft)
n_col_groups = math.ceil(n_tgts_ft / n_cols_ft)

PLOT_H = 2.5
SPACER_H = 1.8
INTER_H = 0.15

height_ratios = []
for g in range(n_col_groups):
    height_ratios.append(PLOT_H)
    if g < n_col_groups - 1:
        height_ratios.append(INTER_H)
        height_ratios.append(PLOT_H)
        height_ratios.append(SPACER_H)
    else:
        height_ratios.append(INTER_H)
        height_ratios.append(PLOT_H)

total_rows = len(height_ratios)
fig_h = sum(height_ratios) + 0.8

fig = plt.figure(figsize=(3.5 * n_cols_ft, fig_h))
gs = GridSpec(
    total_rows,
    n_cols_ft,
    figure=fig,
    height_ratios=height_ratios,
    hspace=0.0,
    wspace=0.35,
)


def gs_row(group, metric_offset):
    return group * (n_metrics_ft + 2) + metric_offset * 2


# ── pre-compute shared ylims per (group_row, metric_offset) ──────────────────
shared_ylims = {}
for tgt_idx, tgt_label in enumerate(tgt_labels_ft):
    group_row = tgt_idx // n_cols_ft
    results = df_ft_metrics_id[tgt_label]
    models = [m for m in FT_MODEL_ORDER if m in results]

    for metric_offset, (metric_key, _, is_r2) in enumerate(METRICS_FT):
        vals = [results[m][metric_key] for m in models]
        valid_vals = [v for v in vals if not np.isnan(v)]
        if not valid_vals:
            continue
        if is_r2:
            ymin = min(min(valid_vals) * 1.3, -0.2)
            ymax = max(max(valid_vals) * 1.3, 0.5)
        else:
            ymin = 0
            ymax = max(valid_vals) * 1.25
        key = (group_row, metric_offset)
        if key not in shared_ylims:
            shared_ylims[key] = (ymin, ymax)
        else:
            shared_ylims[key] = (
                min(shared_ylims[key][0], ymin),
                max(shared_ylims[key][1], ymax),
            )

# ── plot ──────────────────────────────────────────────────────────────────────
for tgt_idx, tgt_label in enumerate(tgt_labels_ft):
    col = tgt_idx % n_cols_ft
    group_row = tgt_idx // n_cols_ft

    results = df_ft_metrics_id[tgt_label]
    models = [m for m in FT_MODEL_ORDER if m in results]
    colors = [FT_COLORS[m] for m in models]

    for metric_offset, (metric_key, metric_label,
                        is_r2) in enumerate(METRICS_FT):
        row = gs_row(group_row, metric_offset)
        ax = fig.add_subplot(gs[row, col])

        panel_idx = group_row * n_cols_ft * n_metrics_ft + metric_offset * n_cols_ft + col
        ax.text(
            0.03,
            0.97,
            f"({chr(ord('a') + panel_idx)})",
            transform=ax.transAxes,
            fontsize=NATURE_SPECS["font_max_pt"],
            #fontweight="bold",
            va="top",
            ha="left",
            zorder=5)

        vals = [results[m][metric_key] for m in models]

        bars = ax.bar(
            range(len(models)),
            vals,
            color=colors,
            edgecolor="white",
            linewidth=0.8,
            alpha=BARPLOT_ALPHA,
            zorder=3,
        )

        for i, mod in enumerate(models):
            if mod == "no_ft":
                bars[i].set_edgecolor("black")
                bars[i].set_linewidth(1.5)
                bars[i].set_alpha(1.0)

        for bar, val in zip(bars, vals):
            if np.isnan(val):
                continue
            y_pos = bar.get_height() + 0.02 if val >= 0 else bar.get_height(
            ) - 0.08
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                y_pos,
                f"{val:.2f}",
                ha="center",
                va="bottom",
                fontsize=8,
            )

        if is_r2:
            ax.axhline(0,
                       color="black",
                       linewidth=0.8,
                       linestyle="--",
                       zorder=2)

        ax.set_xticks(range(len(models)))
        if metric_offset == n_metrics_ft - 1:
            ax.set_xticklabels(
                [FT_LABELS[m] for m in models],
                rotation=90,
                ha="right",
                fontsize=NATURE_SPECS["font_max_pt"],
            )
        else:
            ax.set_xticklabels([])

        if col == 0:
            ax.set_ylabel(metric_label, fontsize=NATURE_SPECS["font_max_pt"])

        if metric_offset == 0:
            ax.set_title(
                REGION_LABELS.get(tgt_label, tgt_label),
                fontsize=NATURE_SPECS["font_max_pt"] + 1,
                fontweight="bold",
                pad=6,
            )

        key = (group_row, metric_offset)
        if key in shared_ylims:
            ax.set_ylim(shared_ylims[key])

        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=False)

fig.suptitle(
    "Foundation model: zero-shot vs fine-tuning strategies (10% FT data)",
    fontsize=NATURE_SPECS["font_max_pt"] + 2,
    y=0.94,
)
fig.savefig(
    "figures/paperTF/fig_pooled_ft_gain_barplot.png",
    dpi=NATURE_SPECS["dpi"],
    bbox_inches="tight",
)
plt.show()

## Distances:

In [ ]:
RECOMPUTE_SHIFTS_POOLED = False
SHIFTS_CACHE_POOLED = BASE_DIR / "domain_shifts_pooled_cache.pkl"

bandwidths_m = [blur_m * 0.5, blur_m, blur_m * 2.0]
bandwidths_s = [blur_s * 0.5, blur_s, blur_s * 2.0]
scaler_joint = scaler_joint  # already computed in scalers cell

EXCLUDE_REGIONS = {'CENTRALASIA', 'ALA'}
tgt_labels = [t for t in df_ft_metrics_id.keys() if t not in EXCLUDE_REGIONS]

if RECOMPUTE_SHIFTS_POOLED and SHIFTS_CACHE_POOLED.exists():
    SHIFTS_CACHE_POOLED.unlink()
    print(f"Deleted existing cache: {SHIFTS_CACHE_POOLED}")

if SHIFTS_CACHE_POOLED.exists():
    with open(SHIFTS_CACHE_POOLED, "rb") as f:
        cache = pickle.load(f)
    all_shifts_pooled = cache["all_shifts_pooled"]
    print(f"Loaded pooled shifts from cache: {SHIFTS_CACHE_POOLED}")

else:
    # df_src = full pooled training data (all source regions combined)
    df_src_pooled = pd.concat(
        [res_xreg_by_source[r]["df_train"] for r in SOURCE_REGIONS],
        ignore_index=True,
    )

    all_shifts_pooled = {}
    for region in tqdm(tgt_labels, desc="Pooled domain shifts"):
        df_tgt = monthly_cache[region]["data_monthly"]
        shift = compute_domain_shift(
            df_src=df_src_pooled,
            df_tgt=df_tgt,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            scaler_m=scaler_m,
            scaler_s=scaler_s,
            scaler_joint=scaler_joint,
            blur_m=blur_m,
            blur_s=blur_s,
            blur_joint=blur_joint,
            bandwidths_m=bandwidths_m,
            bandwidths_s=bandwidths_s,
            seed=cfg.seed,
        )
        all_shifts_pooled[region] = shift
        print(f"  {region}: joint={shift['D_sinkhorn_joint']:.4f} "
              f"clim={shift['D_sinkhorn_climate']:.4f} "
              f"topo={shift['D_sinkhorn_topo']:.4f}")

    with open(SHIFTS_CACHE_POOLED, "wb") as f:
        pickle.dump({"all_shifts_pooled": all_shifts_pooled}, f)
    print(f"Saved pooled shifts to cache: {SHIFTS_CACHE_POOLED}")

fig = plot_domain_shift_across_regions(all_shifts_pooled, src_region="Pooled")
plt.show()

In [ ]:
from scipy import stats

TF_COLOR = NATURE_PALETTE["sky_blue"]


def plot_pooled_performance_vs_shift(
    all_shifts_pooled: dict,
    df_metrics: dict,
    metric: str = "RMSE_annual",
    model_label: str = "no_ft",  # zero-shot entry in df_ft_metrics_id
    distance_metrics: list[str] = [
        "D_sinkhorn_climate", "D_sinkhorn_topo", "D_sinkhorn_joint"
    ],
    model: str = "Transformer",
):
    # ── 1. Build merged DataFrame ─────────────────────────────────────────────
    shift_rows = []
    for tgt, shift in all_shifts_pooled.items():
        row = {"target": tgt}
        row.update(shift)
        shift_rows.append(row)
    df_shifts = pd.DataFrame(shift_rows)

    perf_rows = []
    for tgt, results in df_metrics.items():
        if model_label not in results:
            continue
        row = {"target": tgt, metric: results[model_label][metric]}
        std_key = f"{metric}_std"
        if std_key in results[model_label]:
            row[std_key] = results[model_label][std_key]
        perf_rows.append(row)
    df_perf = pd.DataFrame(perf_rows)

    df_plot = df_shifts.merge(df_perf, on="target")

    # ── 2. Plot ───────────────────────────────────────────────────────────────
    dist_labels = {
        "D_sinkhorn_climate": "Sinkhorn distance (climate)",
        "D_sinkhorn_topo": "Sinkhorn distance (topographic)",
        "D_sinkhorn_joint": "Sinkhorn distance (joint)",
        "D_mmd2_climate": "MMD² (climate)",
        "D_mmd2_topo": "MMD² (topographic)",
        "D_energy_climate": "Energy distance (climate)",
    }

    n = len(distance_metrics)
    fig, axes = plt.subplots(
    1, n,
    figsize=(4.5 * n, 4),
    constrained_layout=False,  # turn off auto-redistribution
    )
    fig.subplots_adjust(left=0.08, right=0.98, bottom=0.18, top=0.85, wspace=0.3)
    if n == 1:
        axes = [axes]

    std_key = f"{metric}_std"
    has_err = std_key in df_plot.columns

    for ax, dist_key in zip(axes, distance_metrics):
        x = df_plot[dist_key].values
        y = df_plot[metric].values
        labels = df_plot["target"].values
        yerr = df_plot[std_key].values if has_err else None

        ax.errorbar(
            x,
            y,
            yerr=yerr,
            fmt="o",
            color=TF_COLOR,
            markersize=8,
            zorder=3,
            ecolor="0.5",
            elinewidth=1.0,
            capsize=3,
            markeredgecolor="white",
            markeredgewidth=0.5,
        )

        for xi, yi, lab in zip(x, y, labels):
            ax.annotate(
                f"Pooled \u2192 {lab}",
                (xi, yi),
                xytext=(5, 4),
                textcoords="offset points",
                fontsize=8,
                color="0.3",
            )

        # leave headroom above the highest point for its annotation
        y_min, y_max = y.min(), y.max()
        y_range = y_max - y_min if y_max > y_min else 1.0
        ax.set_ylim(y_min - y_range * 0.1, y_max + y_range * 0.5)

        # correlation annotation
        if len(x) > 2:
            r, p = stats.pearsonr(x, y)
            # ax.text(
            #     0.05, 0.95,
            #     f"r={r:.2f}, p={p:.3f}",
            #     transform=ax.transAxes,
            #     fontsize=8,
            #     va="top",
            #     ha="left",
            #     bbox=dict(boxstyle="round", facecolor="white", alpha=0.6),
            # )

        ax.set_xlabel(dist_labels.get(dist_key, dist_key), fontsize=9)
        ax.set_ylabel(
            f"{model} RMSE annual (m w.e.)"
            if dist_key == distance_metrics[0] else "",
            fontsize=9,
        )
        #ax.set_title(dist_labels.get(dist_key, dist_key), fontsize=10)

    #fig.suptitle(f"Pooled model: {model} RMSE vs. domain shift", fontsize=11)
    return fig

In [ ]:
fig = plot_pooled_performance_vs_shift(
    all_shifts_pooled,
    df_ft_metrics_id,
    metric="RMSE_annual",
    model_label="no_ft",
)
# save figure
fig.savefig("figures/paperTF/appendix/domain_shift_pooled.png")
plt.show()